<a id='enumerate-zip'></a>

## 13. 🧩 Pattern 13: enumerate() and zip() — Index+Value & Parallel Iteration — LC 48, 238, 344, 986

---

```
PROBLEM:
  LC 48  — Rotate Image: transpose via zip(*matrix) — unzip pattern
  LC 54  — Spiral Matrix: index tracking with enumerate
  LC 56  — Merge Intervals: zip(intervals, intervals[1:]) — adjacent pair scan
  LC 238 — Product of Array Except Self: enumerate for prefix/suffix pass
  LC 344 — Reverse String: zip(s, reversed(s)) — compare mirror pairs
  LC 986 — Interval List Intersections: zip two sorted interval lists

enumerate(iterable, start=0):
  Wraps any iterable — yields (index, value) pairs.
  Replaces the range(len(x)) anti-pattern.
  ✅  for i, v in enumerate(nums)        ← pythonic
  ❌  for i in range(len(nums)): v=nums[i] ← C-style, avoid

  start= shifts the counter:
  for i, v in enumerate(words, start=1)  ← 1-indexed output

zip(a, b, c, ...):
  Pairs up elements across iterables by position.
  Stops at the SHORTEST iterable — extra elements silently dropped.
  for a, b in zip(list1, list2)   ← parallel walk

zip STOPS AT SHORTEST — example:
  zip([1,2,3], [10,20])  →  (1,10), (2,20)   ← 3 dropped silently
  ⚠  Unequal lengths → use zip_longest to keep all.

zip_longest from itertools:
  from itertools import zip_longest
  zip_longest([1,2,3], [10,20], fillvalue=0)  →  (1,10),(2,20),(3,0)

ADJACENT PAIRS — zip(seq, seq[1:]):
  nums = [1, 3, 6, 10]
  for a, b in zip(nums, nums[1:]):
      print(b - a)   # differences: 2, 3, 4
  # slow motion:
  # zip([1,3,6,10], [3,6,10]) → (1,3),(3,6),(6,10)

UNZIP / TRANSPOSE — zip(*matrix):
  matrix = [[1,2,3],[4,5,6],[7,8,9]]
  cols   = list(zip(*matrix))          # → [(1,4,7),(2,5,8),(3,6,9)]
  # zip(*matrix) unpacks rows as separate args — zip then pairs by column
  # This IS the LC 48 transpose.
  as_lists = [list(row) for row in zip(*matrix)]  ← convert tuples to lists

UNZIP PAIRS — split list of tuples:
  pairs  = [(1,'a'),(2,'b'),(3,'c')]
  nums, chars = zip(*pairs)            # → (1,2,3), ('a','b','c')
  # zip(*pairs) is the inverse of zip(nums, chars)

SLOW MOTION TRACE — enumerate(["a","b","c"], start=1):
  step 1: yields (1, "a")
  step 2: yields (2, "b")
  step 3: yields (3, "c")
  Use: for rank, word in enumerate(sorted_words, start=1)

KEY INSIGHT:
  enumerate() = index ticket stapled to every value — never write range(len()) again.
  zip() = parallel conveyor belt — one item from each lane per step.
  zip(*matrix) = rotate conveyor 90° — rows become columns.

TIME / SPACE:
  Time:  O(n) — one pass, lazy
  Space: O(1) — both are iterators; O(n) only after list() materialization
```

In [ ]:
# Pattern 13: enumerate() and zip()
# enumerate = index ticket on every value; zip = parallel conveyor belt.

from itertools import zip_longest

# 1. enumerate — replaces range(len())
words = ["apple", "fig", "banana", "cherry"]

# old C-style — avoid
for i in range(len(words)):
    pass   # words[i] accessed by index

# pythonic
for i, word in enumerate(words):
    pass   # i and word available directly

# with start= for 1-indexed output
for rank, word in enumerate(sorted(words), start=1):
    print(f"  {rank}. {word}")

# 2. zip — parallel iteration
names  = ["alice", "bob", "carol"]
scores = [90, 85, 92]
for name, score in zip(names, scores):
    print(f"  {name}: {score}")

# 3. zip stops at shortest — silent truncation
a = [1, 2, 3]
b = [10, 20]
print(f"zip stops short : {list(zip(a, b))}")        # (1,10),(2,20) — 3 dropped

# zip_longest fills the gap
print(f"zip_longest     : {list(zip_longest(a, b, fillvalue=0))}")  # (1,10),(2,20),(3,0)

# 4. adjacent pairs — zip(seq, seq[1:])
nums = [1, 3, 6, 10, 15]
diffs = [b - a for a, b in zip(nums, nums[1:])]
# slow motion:
# zip([1,3,6,10,15], [3,6,10,15]) → (1,3),(3,6),(6,10),(10,15)
# diffs:                              2,    3,    4,     5
print(f"adjacent diffs  : {diffs}")

# 5. unzip / transpose — zip(*matrix)
matrix = [[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]
transposed = [list(row) for row in zip(*matrix)]
# zip(*matrix) unpacks three rows as separate args:
# zip([1,2,3],[4,5,6],[7,8,9]) → (1,4,7),(2,5,8),(3,6,9)
print(f"transposed:")
for row in transposed:
    print(f"  {row}")

# 6. unzip pairs — inverse of zip
pairs       = [(1, "a"), (2, "b"), (3, "c")]
nums2, chars = zip(*pairs)    # zip(*pairs) splits tuple columns back into separate tuples
print(f"unzipped nums : {nums2}")
print(f"unzipped chars: {chars}")


def rotate_image(matrix: list) -> None:
    """
    LC 48 — Rotate Image
    Approach: transpose via zip(*matrix), then reverse each row in-place.
    Args:
        matrix (list[list[int]]): n×n matrix, modified in-place.
    Returns:
        None — mutates matrix directly.
    Time:  O(n²) — every element touched once
    Space: O(n²) — zip(*matrix) builds transposed rows
    """
    n = len(matrix)
    # step 1: transpose — zip(*matrix) pairs columns into rows
    # slow motion on [[1,2],[3,4]]:
    # zip(*[[1,2],[3,4]]) = zip([1,2],[3,4]) → (1,3),(2,4)
    # transposed: [[1,3],[2,4]]
    transposed = [list(row) for row in zip(*matrix)]

    # step 2: reverse each row — completes 90° clockwise rotation
    # [[1,3],[2,4]] → [[3,1],[4,2]]
    for i in range(n):
        matrix[i] = transposed[i][::-1]


def test_harness(fn):
    import copy
    tests = [
        ([[1,2,3],[4,5,6],[7,8,9]],      [[7,4,1],[8,5,2],[9,6,3]]),
        ([[5,1,9,11],[2,4,8,10],[13,3,6,7],[15,14,12,16]],
         [[15,13,2,5],[14,3,4,1],[12,6,8,9],[16,7,10,11]]),
        ([[1]],                           [[1]]),
    ]
    passed = 0
    for matrix, expected in tests:
        m = copy.deepcopy(matrix)
        fn(m)
        status = "PASSED" if m == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={m}")
        passed += (m == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(rotate_image)

print("enumerate_and_zip defined.")